<center><span style="font-family: TimesNewRoman; font-size:1.4em;color:blue;"><b>  LinearRegression:  Predict Price for Used Car in India</b></span></center><br>
<center><img src=https://images.unsplash.com/photo-1506883968894-6e7738ccfc05?ixid=MnwxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8&ixlib=rb-1.2.1&auto=format&fit=crop&w=800&q=80 width="400" height="300"></center>
<p size=1 ><i>image source:Meriç Dağlı on Unsplash</i></p>
<br>


## Context
    
<p style = "font-size : 15px ; color: black;font-family:TimesNewRoman">
There is a huge demand for used cars in the Indian Market today. As sales of new cars have slowed down in the recent past, the pre-owned car market has continued to grow over the past years and is larger than the new car market now. Cars4U is a budding tech start-up that aims to find footholes in this market.
In 2018-19, while new car sales were recorded at 3.6 million units, around 4 million second-hand cars were bought and sold. There is a slowdown in new car sales and that could mean that the demand is shifting towards the pre-owned market. In fact, some car sellers replace their old cars with pre-owned cars instead of buying new ones. Unlike new cars, where price and supply are fairly deterministic and managed by OEMs (Original Equipment Manufacturer / except for dealership level discounts which come into play only in the last stage of the customer journey), used cars are very different beasts with huge uncertainty in both pricing and supply. Keeping this in mind, the pricing scheme of these used cars becomes important in order to grow in the market. We have to come up with a pricing model that can effectively predict the price of used cars and can help the business in devising profitable strategies using differential pricing.</p>


## Data Set
<br>
<p style = "font-size : 15px ; color: black;font-family:TimesNewRoman">
    
1. S.No. : Serial Number<br>
    
2. Name : Name of the car which includes Brand name and Model name<br>
    
3. Location : The location in which the car is being sold or is available for purchase Cities<b<br>r>
    
4. Year : Manufacturing year of the car<br>
    
5. Kilometers_driven : The total kilometers driven in the car by the previous owner(s) in KM.<br>
    
6. Fuel_Type : The type of fuel used by the car. (Petrol, Diesel, Electric, CNG, LPG)<br>
    
7. Transmission : The type of transmission used by the car. (Automatic / Manual)<br>
    
8. Owner : Type of ownership<br>
    
9. Mileage : The standard mileage offered by the car company in kmpl or km/kg<br>
    
10. Engine : The displacement volume of the engine in CC.<br>
    
11. Power : The maximum power of the engine in bhp.<br>
    
12. Seats : The number of seats in the car.<br>
    
13. New_Price : The price of a new car of the same model in INR Lakhs.(1 Lakh = 100, 000)<br>
    
14. Price : The price of the used car in INR Lakhs (1 Lakh = 100, 000)<br>
</p>


## Part 2: Exploratory Data Analysis & Imputation


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load preprocessed data from Part 1
cars = pd.read_csv('used_cars_preprocessed.csv')

# Convert columns to appropriate categories
for col in ['Location', 'Fuel_Type', 'Transmission', 'Owner_Type', 'Brand_Class']:
    if col in cars.columns:
        cars[col] = cars[col].astype('category')

print('Loaded preprocessed dataset:', cars.shape)


# EDA


In [ ]:
cars.info()


In [ ]:
cars.describe()


<p style = "font-size : 20px ; color: blue;font-family:TimesNewRoman"><b>Observations</b></p>

    
- Years is left skewed. Years ranges from 1996- 2019 . Age of cars 2 year old to 25 years old

- Kilometer driven , median is ~53k Km and mean is ~58K. Max values seems to be 6500000. This is very high , and seems to be outlier. Need to analyze further.

- Mileage is almost Normally distrubuited

- Engine is right skewed and has outliers on higher  and lower end

- Power and Price are also right skewed.

- Price 160 Lakh is too much for a used car. Seems to be an outlier.


In [ ]:
plt.style.use('ggplot')
#select all quantitative columns for checking the spread
numeric_columns = cars.select_dtypes(include=np.number).columns.tolist()
plt.figure(figsize=(20,25))

for i, variable in enumerate(numeric_columns):
                     plt.subplot(10,3,i+1)
                       
                     sns.distplot(cars[variable],kde=False,color='blue')
                     plt.tight_layout()
                     plt.title(variable)


<p style = "font-size : 15px ; color: blue;font-family:TimesNewRoman">
    <b>Observations</b></p>
    
  
- Year is left skewed and has outilers on lower side., This column can be dropped
- Kilometer_driven is right skewed.
- Mileage is almost Normally distrubuted. Has few outliers on upper and lower side. need to check further.
- Engine ,power and price are  right skewed and has outliers on upper side.
- Age of car is right skewed.


In [ ]:
cat_columns=['Location','Fuel_Type','Transmission', 'Owner_Type', 'Brand'] #cars.select_dtypes(exclude=np.number).columns.tolist()

plt.figure(figsize=(15,21))

for i, variable in enumerate(cat_columns):
                     plt.subplot(4,2,i+1)
                     order = cars[variable].value_counts(ascending=False).index    
                     ax=sns.countplot(x=cars[variable], data=cars , order=order ,palette='viridis')
                     for p in ax.patches:
                           percentage = '{:.1f}%'.format(100 * p.get_height()/len(cars[variable]))
                           x = p.get_x() + p.get_width() / 2 - 0.05
                           y = p.get_y() + p.get_height()
                           plt.annotate(percentage, (x, y),ha='center')
                     plt.xticks(rotation=90)
                     plt.tight_layout()
                     plt.title(variable)


<p style = "font-size : 20px ; color: blue;font-family:TimesNewRoman">
    <b>Observations</b></p>
   
   **Car Profile**
    
-  ~71 % cars available for sell have manual Transmission.
- ~82 % cars are First owned cars.
- ~39% of car available for sale are from  Maruti & Hyundai brands.
-  ~53% of car being sold/avialable for purchase  have fuel type as Diesel .
- Mumbai has highest numbers of car availabe for purchase whereas Ahmedabad has least
- Most of the cars are 5 seaters.
- Car being sold/available for purchase are in  2 - 23 years old
- ~ 71% car are lower price range car.


In [ ]:
numeric_columns= numeric_columns = cars.select_dtypes(include=np.number).columns.tolist()
plt.figure(figsize=(13,17))

for i, variable in enumerate(numeric_columns):
                     plt.subplot(5,2,i+1)
                     sns.scatterplot(x=cars[variable],y=cars['Price']).set(title='Price vs '+ variable)
                     #plt.xticks(rotation=90)
                     plt.tight_layout()


# Handling missing values


In [ ]:
cars.isnull().sum()


### Calculating missing values in each row


In [ ]:
# counting the number of missing values per row
num_missing = cars.isnull().sum(axis=1)
num_missing.value_counts()


In [ ]:
#Investigating how many missing values per row are there for each variable
for n in num_missing.value_counts().sort_index().index:
    if n > 0:
        print("*" *30,f'\nFor the rows with exactly {n} missing values, NAs are found in:')
        n_miss_per_col = cars[num_missing == n].isnull().sum()
        print(n_miss_per_col[n_miss_per_col > 0])
        print('\n\n')


This confirms that certain columns tend to be missing together or all nonmissing together. So will try to fill the missing values , as much as possible.


In [ ]:
cars[num_missing==7]


In [ ]:
col=['Engine','Power','Mileage']
cars[col].isnull().sum()


We can start filling missing values by grouping  name and year and fill in missing values. with median.


In [ ]:
cars.groupby(['Name','Year'])['Engine'].median().head(30)


In [ ]:
cars['Engine'] = cars.groupby(['Name', 'Year'])['Engine'].transform(
    lambda x: x.fillna(x.median())
)

cars['Power'] = cars.groupby(['Name', 'Year'])['Power'].transform(
    lambda x: x.fillna(x.median())
)

cars['Mileage'] = cars.groupby(['Name', 'Year'])['Mileage'].transform(
    lambda x: x.fillna(x.median())
)


In [ ]:
col=['Engine','Power','Mileage']
cars[col].isnull().sum()


In [ ]:
cars.groupby(['Brand','Model'])['Engine'].median().head(10)


As we can see most of the model have same engine 
size and instead  of just applying median , grouping with model and year that should give me more granularity, and near to accurate Engine values.


In [ ]:
#chosing Median to fill the the missing value as there are many outliers, 
#grouping by model and year to get  more granularity and more accurate Engine and then fillig with median
cars['Engine'] = cars.groupby(['Brand', 'Model'])['Engine'].transform(
    lambda x: x.fillna(x.median())
)


In [ ]:
#chosing Median to fill the the missing value as there are many outliers, 
#grouping by model to get more granularity and more accurate Engine
cars['Power'] = cars.groupby(['Brand', 'Model'])['Power'].transform(
    lambda x: x.fillna(x.median())
)


In [ ]:
#chosing Median to fill the the missing value as there are many outliers, 
#grouping by model to get more granularity and more accurate Engine
cars['Mileage'] = cars.groupby(['Brand', 'Model'])['Mileage'].transform(
    lambda x: x.fillna(x.median())
)


In [ ]:
col=['Engine','Power','Mileage']
cars[col].isnull().sum()


 There are still missing values , analyzing further .Grouping by only Model for Engine and then filling missing values with median. For  Power and Mileage Engine values for a Brand can be used to get more accurate value.


In [ ]:
cars.groupby(['Model', 'Year'])['Engine'] \
    .agg(['median', 'mean', 'max']) \
    .sort_index() \
    .head(10)


In [ ]:
cars.groupby(['Brand','Engine'])['Power'].agg({'mean','median','max'}).head(10)


In [ ]:
cars['Seats'].isnull().sum()


Grouping with Name should give me more granularity, and near to accurate Seat values.


In [ ]:
cars['Seats'] = cars.groupby('Name')['Seats'].transform(
    lambda x: x.fillna(x.median())
)


In [ ]:
cars['Seats'].isnull().sum()


Grouping with Model should give me more granularity, and near to accurate Seat values.


In [ ]:
cars['Seats'] = cars.groupby('Model')['Seats'].transform(
    lambda x: x.fillna(x.median())
)


In [ ]:
cars['Seats'].isnull().sum()


 Lets check which car types have missing values.


In [ ]:
cars[cars['Seats'].isnull()==True].head(10)


In [ ]:
#most of cars are 5 seater so fillrest of 23 by 5
cars['Seats']=cars['Seats'].fillna(5)


In [ ]:
cars['Seats'].isnull().sum()


Need to analyse along with price if seats plays any role in price


In [ ]:

cars["Location"] = cars["Location"].astype("category")
cars['Brand'] =cars['Brand'].astype("category")


In [ ]:
cars.info()


# Processing New Price


In [ ]:
#For better granualarity grouping has there would be same car model present so filling with a median value brings it more near to real value
cars['new_price_num']=cars.groupby(['Name','Year'])['new_price_num'].transform(lambda x:x.fillna(x.median()))


In [ ]:
cars.new_price_num.isnull().sum()


In [ ]:
cars['new_price_num']=cars.groupby(['Name'])['new_price_num'].transform(lambda x:x.fillna(x.median()))


In [ ]:
cars.new_price_num.isnull().sum()


In [ ]:
cars['new_price_num']=cars.groupby(['Brand','Model'])['new_price_num'].transform(lambda x:x.fillna(x.median()))


In [ ]:
cars.new_price_num.isnull().sum()


In [ ]:
cars['new_price_num']=cars.groupby(['Brand'])['new_price_num'].transform(lambda x:x.fillna(x.median()))


In [ ]:
cars.drop(['New_Price'],axis=1,inplace=True)


In [ ]:
cars.new_price_num.isnull().sum()


In [ ]:
cars.groupby(['Brand'])['new_price_num'].median().sort_values(ascending=False)


In [ ]:
cars.isnull().sum()


In [ ]:

cols1 = ["Power","Mileage","Engine"]

for ii in cols1:
    cars[ii] = cars[ii].fillna(cars[ii].median())


In [ ]:
#dropping remaining rows
#cannot further fill this rows so dropping them

cars.dropna(inplace=True,axis=0)


In [ ]:
cars.isnull().sum()


In [ ]:
cars.head()


In [ ]:
cars.isnull().sum()


In [ ]:
df.shape 


### Missing Values Imputation Complete


In [ ]:
cars.groupby(['Brand'])['Price'].agg({'median','mean','max'})


In [ ]:
#using business knowledge to create class 
Low=['Maruti', 
     'Hyundai',
     'Ambassdor',
     'Hindustan',
     'Force',
     'Chevrolet',
     'Fiat',
     'Tata',
     'Smart',
     'Renault',
     'Datsun',
     'Mahindra',
     'Skoda',
     'Ford',
     'Toyota',
     'Isuzu',
     'Mitsubishi','Honda']

High=['Audi',
      'Mini Cooper',
      'Bentley',
      'Mercedes-Benz',
      'Lamborghini',
      'Volkswagen',
      'Porsche',
      'Land Rover',
      'Nissan',
      'Volvo',
      'Jeep',
      'Jaguar',
      'BMW']# more than 30lakh


In [ ]:
def classrange(x):
    if x in Low:
        return "Low"
    elif x in High:
        return "High"
    else: 
        return x


In [ ]:
cars['Brand_Class'] = cars['Brand'].apply(lambda x: classrange(x))


In [ ]:
cars['Brand_Class'].unique()


In [ ]:
cars['Engine']=cars['Engine'].astype(int)
cars['Brand_Class']=cars["Brand_Class"].astype('category')


### Bivariate & Multivariate Analysis


In [ ]:
plt.figure(figsize=(10, 8))

corr_matrix = cars.corr(numeric_only=True)

sns.heatmap(corr_matrix, annot=True, cmap="YlGnBu")

plt.show()


<p style = "font-size : 20px ; color: blue;font-family:TimesNewRoman">
    <b>Observations</b></p>

    
- Engine has strong positive correlation to Power [0.86]. 
- Price has positive correlation to Engine[0.66] as well Power [0.77].
- Mileage is negative correlated to Engine,Power,Price.,Ageofcar
- Price has negative  correlation to age of car.
- Kilometer driven doesnt impact Price


In [ ]:
sns.pairplot(data=cars , corner=True)
plt.show()


<p style = "font-size : 15px ; color: blue;font-family:TimesNewRoman">
    <b>Observations</b></p>
    
- Same observation  about correlation as seen in heatmap.

- Kilometer driven  doesnot have impact on  Price . 
- As power increase mileage decrease.
- Car with recent make sell at higher prices.
- Engine and Power increase , price of the car seems to increase.


### Variables that are correlated with Price variable


#### Price  Vs Engine Vs Transmission


In [ ]:
# understand relation ship of Engine vs Price and Transmimssion
plt.figure(figsize=(10,7))

plt.title("Price VS Engine based on Transmission")
sns.scatterplot(y='Engine', x='Price', hue='Transmission', data=cars)


#### Price Vs Power vs Transmission


In [ ]:
 #understand relationship betweem Price and Power
plt.figure(figsize=(10,7))
plt.title("Price vs Power based on Transmission")
sns.scatterplot(y='Power', x='Price', hue='Transmission', data=cars)


#### Price Vs Mileage Vs Transmission


In [ ]:
# Understand the relationships  between mileage and Price
sns.scatterplot(y='Mileage', x='Price', hue='Transmission', data=cars)


#### Price Vs Year Vs Transmission


In [ ]:
# Impact of years on price 
plt.figure(figsize=(10,7))
plt.title("Price based on manufacturing Year of model")
sns.lineplot(x='Year', y='Price',hue='Transmission',
             data=cars)


#### Price Vs Year VS Fuel Type


In [ ]:
# Impact of years on price 
plt.figure(figsize=(10,7))
plt.title("Price Vs Year VS FuelType")
sns.lineplot(x='Year', y='Price',hue='Fuel_Type',
             data=cars)


#### Year Vs Price Vs Owner_Type


In [ ]:
plt.figure(figsize=(10,7))
plt.title("Price Vs Year VS Owner_Type")
sns.lineplot(x='Year', y='Price',hue='Owner_Type',
             data=cars)


Need to check the reason for spike in price  for third owner and model in 2010.


In [ ]:
cars[(cars["Owner_Type"]=='Third') & (cars["Year"].isin([2010]))].sort_values(by='Price',ascending =False)


The observation is for The Porsche Panamera is expensive and luxury car so the data is valid.


In [ ]:
cars.describe()


#### Price Vs Mileage vs Fuel_type


In [ ]:
# Understand relationships  between price and mileage
plt.figure(figsize=(10,7))
plt.title("Price Vs Mileage")
sns.scatterplot(y='Price', x='Mileage', hue='Fuel_Type', data=cars)


#### Price Vs Seat


In [ ]:
#Price and seats 
plt.figure(figsize=(20,15))
sns.set(font_scale=2)
sns.barplot(x='Seats', y='Price', data=cars)
plt.grid()


#### Price Vs Location


In [ ]:
#Price and LOcation 
plt.figure(figsize=(20,15))
sns.set(font_scale=2)
sns.barplot(x='Location', y='Price', data=cars)
plt.grid()


#### Price Vs Brand


In [ ]:
#Price and band 
plt.figure(figsize=(20,15))
sns.set(font_scale=2)
sns.boxplot(x='Price', y='Brand', data=cars)
plt.grid()


In [ ]:
sns.relplot(data=cars, y='Price',x='Mileage',hue='Transmission',aspect=1,height=5)


In [ ]:
sns.relplot(data=cars, y='Price',x='Year',col='Owner_Type',hue='Transmission',aspect=1,height=5)


In [ ]:
sns.relplot(data=cars, y='Price',x='Engine',col='Transmission',aspect=1,height=6,hue="Fuel_Type")


In [ ]:

sns.relplot(data=cars, y='Price',x='Ageofcar',col='Transmission',aspect=1,height=6)


# Insights based on EDA


<p style = "font-size : 20px ; color: blue;font-family:TimesNewRoman">
<b>Observations</b>
</p>


- Expensive cars are in Coimbatore and Banglore.
- 2 Seater cars are more expensive.
- Deisel Fuel type car are more expensive compared to other fuel type.
- As expected, Older model are sold cheaper compared to latest model
- Automatic transmission vehicle have a higher price than manual transmission vehicles.
- Vehicles with more engine capacity have higher prices. 
- Price decreases as number of owner increases.
- Automatic transmission require high engine and power.
- Prices for Cars with fuel type as Deisel has increased with recent models 
- Engine,Power, how old the car his, Mileage,Fuel type,location,Transmission effect the price.


In [ ]:
# check distrubution if skewed. If distrubution is skewed , it is advice to use log transform
cols_to_log = cars.select_dtypes(include=np.number).columns.tolist()
for colname in cols_to_log:
    sns.distplot(cars[colname], kde=True)
    plt.show()


Distrubtions are right skewed , using Log transform can help in normalization


In [ ]:

def Perform_log_transform(df,col_log):
    """#Perform Log Transformation of dataframe , and list of columns """
    for colname in col_log:
        df[colname + '_log'] = np.log(df[colname])
    #df.drop(col_log, axis=1, inplace=True)
    df.info()


In [ ]:
#This needs to be done before the data is split
Perform_log_transform(cars,['Kilometers_Driven','Price'])


In [ ]:

cars.drop(['Name','Model','Year','Brand','new_price_num'],axis=1,inplace=True)


In [ ]:
cars.info()


In [ ]:
# Save fully processed and imputed data for Model Building
cars.to_csv('used_cars_imputed.csv', index=False)
print('Imputed dataset successfully saved to used_cars_imputed.csv')
